# Simple: Fake Organization Generation with PAMOLA.CORE

**Goal**: Learn synthetic organization name generation basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Generate synthetic organization names using execute()
- Compare before/after organization distributions
- Understand privacy protection through organization pseudonymization

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── fake_data/
        └── 01_fake_organization_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.fake_data.operations.organization_op import FakeOrganizationOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with business records containing real organization names

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample company data with realistic organization names
    sample_data = pd.DataFrame({
        'company_id': [f'COMP{i:04d}' for i in range(1, 51)],
        'company_current': [
            'Acme Corporation', 'Global Tech Solutions', 'Smith & Associates LLC',
            'First National Bank', 'Johnson Medical Center', 'Pacific Airlines Inc',
            'Metro Transportation Authority', 'Davis Construction Co', 'Wilson Electronics',
            'Anderson Manufacturing', 'Taylor Insurance Group', 'Brown Consulting Services',
            'Miller Software Systems', 'Moore Financial Advisors', 'Jackson Real Estate',
            'White Pharmaceuticals Ltd', 'Harris Engineering Corp', 'Martin Retail Stores',
            'Thompson Legal Services', 'Garcia Food Products', 'Martinez Automotive Parts',
            'Robinson Energy Solutions', 'Clark Healthcare Systems', 'Rodriguez Media Group',
            'Lewis Technology Partners', 'Lee Investment Management', 'Walker Communications',
            'Hall Publishing House', 'Allen Security Services', 'Young Educational Institute',
            'King Architecture Firm', 'Wright Marketing Agency', 'Lopez Manufacturing LLC',
            'Hill Transportation Services', 'Green Environmental Solutions', 'Adams Research Laboratory',
            'Baker Hotel Group', 'Nelson Logistics Inc', 'Carter Financial Group',
            'Mitchell Aerospace Corporation', 'Perez Biotechnology Labs', 'Roberts Data Analytics',
            'Turner Construction & Development', 'Phillips Telecommunications', 'Campbell Furniture Co',
            'Parker Industrial Equipment', 'Evans Medical Supplies', 'Edwards Technology Solutions',
            'Collins International Trading', 'Stewart Property Management'
        ],
        'organization_type': [
            'corporation', 'tech', 'llc', 'bank', 'medical', 'airline',
            'government', 'construction', 'electronics', 'manufacturing',
            'insurance', 'consulting', 'software', 'financial', 'realestate',
            'pharmaceutical', 'engineering', 'retail', 'legal', 'food',
            'automotive', 'energy', 'healthcare', 'media', 'tech',
            'investment', 'communications', 'publishing', 'security', 'education',
            'architecture', 'marketing', 'manufacturing', 'transportation', 'environmental',
            'research', 'hospitality', 'logistics', 'financial', 'aerospace',
            'biotech', 'tech', 'construction', 'telecommunications', 'furniture',
            'industrial', 'medical', 'tech', 'trading', 'realestate'
        ],
        'region': ['en'] * 50,
        'industry': [
            'Manufacturing', 'Technology', 'Professional Services', 'Finance', 'Healthcare', 'Transportation',
            'Government', 'Construction', 'Electronics', 'Manufacturing',
            'Insurance', 'Consulting', 'Technology', 'Finance', 'Real Estate',
            'Pharmaceuticals', 'Engineering', 'Retail', 'Legal', 'Food & Beverage',
            'Automotive', 'Energy', 'Healthcare', 'Media', 'Technology',
            'Finance', 'Telecommunications', 'Publishing', 'Security', 'Education',
            'Architecture', 'Marketing', 'Manufacturing', 'Transportation', 'Environmental',
            'Research', 'Hospitality', 'Logistics', 'Finance', 'Aerospace',
            'Biotechnology', 'Technology', 'Construction', 'Telecommunications', 'Manufacturing',
            'Industrial', 'Healthcare', 'Technology', 'Trading', 'Real Estate'
        ],
        'employee_count': np.random.randint(10, 5000, 50),
        'founded_year': np.random.randint(1950, 2020, 50)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
        if col == 'company_current':
            print(f"    Sample: {', '.join(df[col].head(3).tolist())}")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for synthetic organization generation

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "company_current"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "company_current"  # Field to generate synthetic organizations for
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique organizations: {df[field_name].nunique()}")
print(f"  Sample organizations:")
for i, org in enumerate(df[field_name].head(5), 1):
    print(f"    {i}. {org}")
print(f"  Average name length: {df[field_name].str.len().mean():.1f} characters")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_org_001",
    task_type="fake_organization_generation",
    description="Generate synthetic organization names for company data",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Generating synthetic organizations for {field_name}",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create FakeOrganizationOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name='field_name'`: Field containing organization names to replace
- `mode='REPLACE'`: Modify original field (or 'ENRICH' for new field)
- `organization_type='general'`: Type of organizations to generate
- `region='en'`: Region for name generation (English)
- `preserve_type=True`: Preserve organization type characteristics
- `type_field='organization_type'`: Use type information for contextual generation
- `add_prefix_probability=0.3`: 30% chance to add prefix (e.g., "Global")
- `add_suffix_probability=0.5`: 50% chance to add suffix (e.g., "Inc", "LLC")
- `consistency_mechanism='mapping'`: Deterministic generation
- `collect_type_distribution=True`: Collect organization type statistics
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = FakeOrganizationOperation(
    # Core parameters
    field_name=field_name,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='synthetic_',
    output_field_name=None,
    output_format='csv',
    
    # Organization generation parameters
    organization_type='general',      # General business organizations
    region='en',                      # English region
    preserve_type=True,               # Preserve organization types
    type_field='organization_type',   # Use type field for context
    region_field='region',            # Use region field if available
    industry=None,                    # No specific industry filter
    
    # Prefix/Suffix configuration
    add_prefix_probability=0.3,       # 30% chance of prefix
    add_suffix_probability=0.5,       # 50% chance of suffix
    
    # Metrics and monitoring
    collect_type_distribution=True,   # Collect organization type stats
    detailed_metrics=True,            # Enable detailed metrics
    max_retries=3,                    # Retry failed generations
    
    # Consistency and privacy
    consistency_mechanism='mapping',  # Use mapping for consistent replacement
    key='my-secret-key-2025',         # Encryption key for PRGN
    context_salt='company-names',     # Additional randomization salt
    save_mapping=True,                # Save name mappings
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:                    {operation.field_name}")
print(f"  Organization type:        {operation.organization_type}")
print(f"  Region:                   {operation.region}")
print(f"  Preserve type:            {operation.preserve_type}")
print(f"  Type field:               {operation.type_field}")
print(f"  Prefix probability:       {operation.add_prefix_probability:.0%}")
print(f"  Suffix probability:       {operation.add_suffix_probability:.0%}")
print(f"  Consistency:              {operation.consistency_mechanism}")
print(f"  Collect type dist:        {operation.collect_type_distribution}")
print(f"  Detailed metrics:         {operation.detailed_metrics}")
print(f"  Visualizations:           {operation.generate_visualization}")
print(f"  Save output:              {operation.save_output}")
print(f"  Save mapping:             {operation.save_mapping}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing Fake Organization Generation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the generated output from task_dir
- Compare with original data
- Review organization generation metrics and artifacts

**Generated artifacts:**
- Synthetic organization names CSV in output/
- Organization mapping JSON in maps/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records with before/after comparison
    print("\n🔍 Sample Organization Transformations (First 15):")
    print(f"\n{'Original Organization':<40} → {'Synthetic Organization':<40} {'Type':<15}")
    print("-" * 97)
    for i in range(min(15, len(df))):
        orig_org = df[field_name].iloc[i]
        synth_org = result_df[field_name].iloc[i]
        org_type = df['organization_type'].iloc[i] if 'organization_type' in df.columns else 'N/A'
        print(f"{orig_org:<40} → {synth_org:<40} {org_type:<15}")
    
    # Organization statistics comparison
    print(f"\n📊 Organization Statistics Comparison:")
    print("-" * 80)
    
    orig_lengths = df[field_name].str.len()
    synth_lengths = result_df[field_name].str.len()
    
    orig_word_counts = df[field_name].str.split().str.len()
    synth_word_counts = result_df[field_name].str.split().str.len()
    
    print(f"\n  Original Organizations:")
    print(f"    Total unique:     {df[field_name].nunique()}")
    print(f"    Avg length:       {orig_lengths.mean():.1f} chars")
    print(f"    Avg word count:   {orig_word_counts.mean():.1f} words")
    print(f"    Min length:       {orig_lengths.min()} chars")
    print(f"    Max length:       {orig_lengths.max()} chars")
    
    print(f"\n  Synthetic Organizations:")
    print(f"    Total unique:     {result_df[field_name].nunique()}")
    print(f"    Avg length:       {synth_lengths.mean():.1f} chars")
    print(f"    Avg word count:   {synth_word_counts.mean():.1f} words")
    print(f"    Min length:       {synth_lengths.min()} chars")
    print(f"    Max length:       {synth_lengths.max()} chars")
    
    # Organization type distribution
    if 'organization_type' in result_df.columns:
        print(f"\n  Organization Type Distribution:")
        type_counts = result_df['organization_type'].value_counts().head(10)
        for org_type, count in type_counts.items():
            pct = (count / len(result_df)) * 100
            print(f"    {org_type:20s}: {count:3d} ({pct:5.1f}%)")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:      {len(df)}")
    print(f"  Generated records:     {len(result_df)}")
    print(f"  Unique original:       {df[field_name].nunique()}")
    print(f"  Unique synthetic:      {result_df[field_name].nunique()}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:15]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:40s}: {value:.4f}")
                else:
                    print(f"  {key:40s}: {value}")
    
    print(f"\n💡 Organization Generation Insight:")
    print(f"   Synthetic organization names protect business identity while")
    print(f"   preserving organization type characteristics. The PRGN mechanism")
    print(f"   ensures consistent mapping for reproducibility and auditing.")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Synthetic organizations CSV
├── maps/             # Organization mapping JSON
├── metrics/          # Generation metrics
├── reports/          # Detailed reports
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'maps', 'metrics', 'reports', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Organization generation performance and effectiveness
2. **Output Data**: Synthetic organizations with sample rows
3. **Organization Mapping**: Original → Synthetic organization mappings
4. **Visualizations**: Charts showing organization distribution and statistics

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # -------------------------
            # METADATA
            # -------------------------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name:      {metadata.get('name', 'N/A')}")

            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # -------------------------
            # PERFORMANCE
            # -------------------------
            if 'performance' in metrics:
                print("\n⚡ PERFORMANCE:")
                perf = metrics['performance']
                print(f"   Generation Time:   {perf.get('generation_time', 0):.4f}s")
                print(f"   Records/Second:    {perf.get('records_per_second', 0):,}")
                print(f"   Memory Usage:      {perf.get('memory_usage_mb', 0):.2f} MB")

            # -------------------------
            # ORIGINAL DATA STATS
            # -------------------------
            if 'original_data' in metrics:
                print("\n📘 ORIGINAL DATA:")
                od = metrics['original_data']
                
                print(f"   Total Records:     {od.get('total_records', 0):,}")
                print(f"   Unique Values:     {od.get('unique_values', 0):,}")

                vd = od.get("value_distribution", {})
                if vd:
                    print("\n   Top 10 Values:")
                    for k, v in list(vd.items())[:10]:
                        print(f"      {k:<35} {v:.4%}")

            # -------------------------
            # GENERATED DATA STATS
            # -------------------------
            if 'generated_data' in metrics:
                print("\n📗 GENERATED DATA:")
                gd = metrics['generated_data']
                
                print(f"   Total Records:     {gd.get('total_records', 0):,}")
                print(f"   Unique Values:     {gd.get('unique_values', 0):,}")

                vd = gd.get("value_distribution", {})
                if vd:
                    print("\n   Top 10 Synthetic Values:")
                    for k, v in list(vd.items())[:10]:
                        print(f"      {k:<35} {v:.4%}")

                ls = gd.get("length_stats", {})
                if ls:
                    print("\n   Length Statistics:")
                    print(f"      Min:            {ls.get('min', 0)}")
                    print(f"      Max:            {ls.get('max', 0)}")
                    print(f"      Mean:           {ls.get('mean', 0):.3f}")
                    print(f"      Median:         {ls.get('median', 0)}")

            # -------------------------
            # TRANSFORMATION METRICS
            # -------------------------
            if 'transformation_metrics' in metrics:
                print("\n🔄 TRANSFORMATION METRICS:")
                tm = metrics['transformation_metrics']
                print(f"   Null Values Replaced:  {tm.get('null_values_replaced', 0)}")
                print(f"   Total Replacements:    {tm.get('total_replacements', 0)}")
                print(f"   Collisions:            {tm.get('mapping_collisions', 0)}")
                print(f"   Reversibility Rate:    {tm.get('reversibility_rate', 0):.4f}")
                print(f"   Distribution Similar.: {tm.get('distribution_similarity_score', 0):.4f}")
                print(f"   Uniqueness Preserv.:   {tm.get('uniqueness_preservation', 0):.4f}")

            # -------------------------
            # GENERATOR INFO
            # -------------------------
            if 'generator' in metrics:
                print("\n🔧 GENERATOR:")
                gen = metrics['generator']
                print(f"   Type:                 {gen.get('type', 'N/A')}")
                print(f"   Consistency:          {gen.get('consistency_mechanism', 'N/A')}")

            # -------------------------
            # MAPPING
            # -------------------------
            if 'mapping' in metrics:
                print("\n🗺️  MAPPING:")
                mapping = metrics['mapping']
                print(f"   Total Mappings:       {mapping.get('total_mappings', 0):,}")

            # -------------------------
            # ORGANIZATION GENERATOR
            # -------------------------
            if 'organization_generator' in metrics:
                og = metrics['organization_generator']
                print("\n🏢 ORGANIZATION GENERATOR:")

                print(f"   Org Type:             {og.get('organization_type', 'N/A')}")
                print(f"   Region:               {og.get('region', 'N/A')}")
                print(f"   Preserve Type:        {og.get('preserve_type', False)}")
                print(f"   Prefix Prob.:         {og.get('add_prefix_probability', 0):.0%}")
                print(f"   Suffix Prob.:         {og.get('add_suffix_probability', 0):.0%}")

                # Type distribution
                td = og.get('type_distribution', {})
                if td:
                    print("\n   Type Distribution:")
                    print(f"      Total:             {td.get('total_organizations', 0):,}")
                    print(f"      Unique Types:      {td.get('unique_organization_types', 0)}")
                    print(f"      Diversity Ratio:   {td.get('diversity_ratio', 0):.4f}")

                    top = td.get("top_organization_types", {})
                    if top:
                        print("\n      Top Organization Types:")
                        for k, v in list(top.items())[:5]:
                            print(f"         {k:<20} {v:.2%}")

                # Region Distribution
                rd = og.get('region_distribution', {})
                if rd:
                    print("\n   Region Distribution:")
                    print(f"      Total:             {rd.get('total_organizations', 0):,}")
                    print(f"      Unique Regions:    {rd.get('unique_regions', 0)}")

                    reg = rd.get("region_distribution", {})
                    if reg:
                        for k, v in reg.items():
                            print(f"         {k:<20} {v:.2%}")

                # Prefix/Suffix
                ps = og.get("prefix_suffix_distribution", {})
                if ps:
                    psd = ps.get("prefix_suffix_distribution", {})
                    print("\n   Prefix/Suffix Distribution:")
                    print(f"      With Prefix:       {psd.get('with_prefix', 0):.2%}")
                    print(f"      With Suffix:       {psd.get('with_suffix', 0):.2%}")
                    print(f"      With Both:         {psd.get('with_both', 0):.2%}")
                    print(f"      With Neither:      {psd.get('with_neither', 0):.2%}")
                    print(f"      Total Analyzed:    {ps.get('total_analyzed', 0)}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")

            display_cols = [field_name]
            if 'organization_type' in output_df.columns:
                display_cols.append('organization_type')
            if 'industry' in output_df.columns:
                display_cols.append('industry')
            
            print(f"\n{output_df[display_cols].head(10).to_string()}")
            
            # Show organization statistics
            print(f"\n\n📊 {field_name.replace('_', ' ').title()} Statistics:")
            print("-" * 80)
            print(f"   Unique Organizations:   {output_df[field_name].nunique():,}")
            print(f"   Null Values:            {output_df[field_name].isna().sum()}")
            print(f"   Avg Name Length:        {output_df[field_name].str.len().mean():.1f} chars")
            print(f"   Min Name Length:        {output_df[field_name].str.len().min()} chars")
            print(f"   Max Name Length:        {output_df[field_name].str.len().max()} chars")
            print(f"   Avg Word Count:         {output_df[field_name].str.split().str.len().mean():.1f} words")
            
            # Show sample organizations
            print(f"\n📋 Sample Generated Organizations:")
            for i, org in enumerate(output_df[field_name].head(10), 1):
                print(f"      {i:2d}. {org}")
            
            # Show organization type breakdown
            if 'organization_type' in output_df.columns:
                print(f"\n📊 Organization Type Breakdown:")
                type_counts = output_df['organization_type'].value_counts().head(10)
                for org_type, count in type_counts.items():
                    pct = (count / len(output_df)) * 100
                    print(f"      {org_type:20s}: {count:3d} ({pct:5.1f}%)")
            
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. ORGANIZATION MAPPING (NEWEST)
print("\n\n3️⃣ ORGANIZATION MAPPING:")
print("-" * 80)

maps_dir = task_dir / 'maps'
if not maps_dir.exists():
    print(f"⚠️  Maps directory not found: {maps_dir}")
else:
    mapping_files = list(maps_dir.glob('*_mapping.json'))
    
    if not mapping_files:
        print("⚠️  No mapping files found")
    else:
        # Sort by modification time (newest first)
        mapping_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_mapping_file = mapping_files[0]
        
        print(f"✓ Found {len(mapping_files)} mapping file(s)")
        print(f"📄 Loading LATEST: {latest_mapping_file.name}")
        mtime = datetime.fromtimestamp(latest_mapping_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_mapping_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            with open(latest_mapping_file, 'r', encoding='utf-8') as f:
                mapping_data = json.load(f)
            
            # Check mappings
            if 'mappings' in mapping_data:
                all_mappings = mapping_data['mappings']
                print(f"📋 Mapping Summary:")
                print(f"   Total Fields:           {len(all_mappings)}")
                
                if field_name in all_mappings:
                    field_mappings = all_mappings[field_name]   # <-- list of objects
                    print(f"   Mappings for '{field_name}': {len(field_mappings)}")

                    print(f"\n📝 Sample Organization Mappings (first 15):\n")
                    print(f"   {'Original Organization':<40} → {'Synthetic Organization':<40}")
                    print("   " + "-" * 82)

                    # Iterate list of mapping items (not dict)
                    for item in field_mappings[:15]:
                        orig = item.get("original", "")
                        synth = item.get("synthetic", "")
                        print(f"   {orig:<40} → {synth:<40}")

                    if len(field_mappings) > 15:
                        print(f"\n   ... and {len(field_mappings) - 15} more mappings")

                    # ---- Statistics ----
                    orig_lengths = [len(m['original']) for m in field_mappings]
                    synth_lengths = [len(m['synthetic']) for m in field_mappings]

                    print(f"\n📊 Mapping Statistics:")
                    print(f"   Avg Original Length:    {sum(orig_lengths)/len(orig_lengths):.1f} chars")
                    print(f"   Avg Synthetic Length:   {sum(synth_lengths)/len(synth_lengths):.1f} chars")
                    print(f"   Length Preservation:    {(sum(synth_lengths)/sum(orig_lengths)):.2%}")
                else:
                    print(f"⚠️ Field '{field_name}' not found in mapping file.")

            else:
                print("⚠️  No mappings found in file")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading mapping: {e}")

# 4. VISUALIZATIONS (NEWEST BATCH)
print("\n\n4️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                if 'length_distribution' in viz_file.name.lower():
                    viz_name = "Organization Name Length Distribution"
                elif 'type_distribution' in viz_file.name.lower():
                    viz_name = "Organization Type Distribution"
                elif 'statistics' in viz_file.name.lower():
                    viz_name = "Organization Statistics Overview"
                elif 'distribution' in viz_file.name.lower():
                    viz_name = "Organization Distribution Analysis"
                elif 'word_count' in viz_file.name.lower():
                    viz_name = "Organization Name Word Count Analysis"
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/ with real organization names  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with organization generation config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review synthetic organization generation metrics and artifacts

**Key takeaways:**
- `operation.execute()` handles end-to-end organization name generation
- Synthetic organizations protect business identity while maintaining realism
- Type-aware generation ensures appropriate organization characteristics
- PRGN consistency mechanism guarantees deterministic mapping
- Field name is configurable via `create_config_kwargs()`
- Organization mappings are saved for reproducibility and auditing
- All artifacts saved in structured directories  

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_fake_organization_advanced.ipynb`** includes:
- Multiple region support (English, European, Asian naming conventions)
- Custom organization dictionaries
- Industry-specific organization generation
- Complex organization structures (with subsidiaries, divisions)
- Multi-language organization names
- Custom prefix/suffix patterns
- Organization hierarchy preservation
- Advanced type classification
- Cross-table organization consistency

**Other Fake Data Operations:**
- 📗 **`01_fake_name_simple.ipynb`**: Generate synthetic person names
- 📙 **`01_fake_email_simple.ipynb`**: Generate synthetic email addresses
- 📕 **`01_fake_phone_simple.ipynb`**: Generate synthetic phone numbers

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Fake Data Generators Guide](../../docs/fake_data_guide.md)
- 🔒 [PRGN Consistency Guide](../../docs/prgn_mechanism.md)
- 🏢 [Organization Generator Details](../../docs/generators/organization_generator.md)